# ML Problem Framing

The exact ML problem: what is predicted, for which unit, at what moment, and how success is measured, built on the business framing (`01_business_framing.ipynb`) and the data discovery (`02_data_discovery.ipynb`).

**How to use this notebook.** The cells marked *Decision* hold the framing. Edit them, then choose Run All. The *Checks Against the Data* cells test the framing on the data, and the last cells write it to `ml_spec.yaml`. Raw rows are never shown here.

## Setup

Finds the project root, loads the shared spec `ml_spec.yaml`, and checks the kernel. Run this cell first.

In [1]:
import os
import sys
from pathlib import Path

# The project root is the closest folder with ml_spec.yaml. Everything runs from there.
here = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in here if (p / "ml_spec.yaml").is_file()), Path.cwd())
os.chdir(ROOT)
# The project's own code (src) comes from this folder, also in an experiment's worktree, where the editable
# install in .venv would point to the main project.
sys.path.insert(0, str(ROOT))

# Shared code from the ML project skills
sys.path.insert(0, r"/Users/ani/.claude/skills/ml-problem-framing/scripts")
import ml_common
import nb_common
try:  # MLflow tracking, in the ML project skills: keeps a copy of each chart for the stage's run
    import tracking
    tracking.capture_figures()
except ImportError:
    tracking = None

SPEC_PATH = str(ROOT / "ml_spec.yaml")
spec = ml_common.migrate(ml_common.load(SPEC_PATH)) if os.path.exists(SPEC_PATH) else ml_common.template()
PATHS = ml_common.project_paths(spec, str(ROOT))
NOTEBOOK = os.path.join(PATHS["notebooks"], "03_problem_framing.ipynb")
STAGE = "framing"

print("Project root:", ROOT)
print("Python:", sys.executable)
if (ROOT / ".venv").is_dir() and Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    print("Warning: this kernel is not the project's .venv. Choose 'Select Kernel' and pick the project kernel.")

Project root: /Users/ani/Projects/16_weather_prediction
Python: /Users/ani/Projects/16_weather_prediction/.venv/bin/python


In [2]:
import re

import pandas as pd
from IPython.display import display

from nb_common import load_table as load_data

## What This Builds On

* **Mode:** exploratory, so V1 is compared with the best simple baseline, not a current approach
* **Learning question:** Can I build an end to end online inference system (feature store, champion/challenger gate, monitoring) that serves next day maximum temperature forecasts for 50 cities, with a model that beats a simple persistence forecast (tomorrow's maximum equals the last complete day's maximum)?
* **What each answer leads to:** The model clearly beats persistence (for example 10% or more lower error). then Promote it as champion, build the full serving stack around it, and use the gate to judge later challengers.; The model only matches persistence. then Still build the system, but serve the persistence forecast as the first champion, then try richer features (lags from nearby cities, hourly data) as challengers.; The model is worse than persistence. then Look for leakage or a flawed setup first, and ship nothing until it is explained.
* **Outcome in business terms:** Each city's maximum air temperature (2 m, degrees Celsius) for its next local calendar day, predicted whenever a request arrives, from that city's weather up to and including the last complete day (normally yesterday).
* **Output use:** quantity. **V1 bar intent:** Beat the best simple baseline (persistence from the last complete day) by at least 10% lower error on CV, then promote the model as champion and build the serving stack around it.
* **Main source:** weather, 50,250 rows, grain `date` + `city`, 1,004 days of history in `date`
* **Candidate targets from discovery:** `wind_speed_10m_max`, `wind_gusts_10m_max`, `wind_direction_10m_dominant`, `city`, `weather_code`

The cell below prints the same from the spec, so it stays current when the earlier notebooks change.

In [3]:
business, discovery = spec.get("business") or {}, spec.get("discovery") or {}
keys = (("mode", "learning_question", "answer_actions") if business.get("mode") == "exploratory" else
        ("business_goal", "decision_supported", "capacity", "current_approach"))
for key in keys + ("outcome", "output_use", "v1_bar_intent"):
    print(f"{key}: {business.get(key)}")
print(f"main source: {discovery.get('main_source')}")
print("candidate targets: " + ", ".join(c["column"] for c in (discovery.get("framing_hints") or {}).get("candidate_targets", [])))

mode: exploratory
learning_question: Can I build an end to end online inference system (feature store, champion/challenger gate, monitoring) that serves next day maximum temperature forecasts for 50 cities, with a model that beats a simple persistence forecast (tomorrow's maximum equals the last complete day's maximum)?
answer_actions: [{'answer': 'The model clearly beats persistence (for example 10% or more lower error).', 'action': 'Promote it as champion, build the full serving stack around it, and use the gate to judge later challengers.'}, {'answer': 'The model only matches persistence.', 'action': 'Still build the system, but serve the persistence forecast as the first champion, then try richer features (lags from nearby cities, hourly data) as challengers.'}, {'answer': 'The model is worse than persistence.', 'action': 'Look for leakage or a flawed setup first, and ship nothing until it is explained.'}]
outcome: Each city's maximum air temperature (2 m, degrees Celsius) for its 

In [4]:
framing = {}  # filled in by the decision cells below

## Target

The learning question asks for next day maximum temperature, and temperature_2m_max exists for all 50 cities with no gaps (discovery). In the forecasting layout the target is the column itself on the target date; the two day horizon keeps the inputs at the last complete day.

*The exact rule for the outcome, with its window. When `exists_in_data` is false, `construction` says how to build it. task_type is one of binary_classification, multiclass_classification, regression, ranking, forecasting, or other.*

In [5]:
# Decision
framing.update({
    'task_type': 'forecasting',
    'target': {
        'name': 'temperature_2m_max',
        'definition': "The city's maximum air temperature at 2 m (degrees Celsius) on the target date, its local calendar day.",
        'positive_class': None,
        'outcome_window': 'The single target day, two days after the last day the inputs may use',
        'exists_in_data': True,
        'construction': None,
        'label_delay': '1 day (known once the target day is complete and in the archive)',
    },
})

## Unit of Prediction and Population

Discovery found the grain is date plus city, unique, 50 cities by 1,004 days. Each row is one city on one target date; each city has many rows over time, so the split must be by time.

*What one row represents when a prediction is made, the columns that identify the entity, and who is scored.*

In [6]:
# Decision
framing.update({
    'unit_of_prediction': "One city on one target date (the city's next local day, seen from a request)",
    'entity_id_cols': ['city'],
    'population': 'All 50 cities in the snapshot, every day',
})

## Prediction Time

Requests can arrive at any time, when the current day is incomplete, so inputs stop at the last complete day. Seen from the target date that is two days earlier, hence a horizon of 2 days, which feature engineering enforces through min_shift_days.

*When the prediction is made, the column that marks that moment, and the latest data a feature may use.*

In [7]:
# Decision
framing.update({
    'prediction_time': {
        'description': 'Whenever a request arrives during the day before the target date. The current day is incomplete, so inputs stop at the last complete day, two days before the target date.',
        'column': 'date',
        'feature_cutoff': 'End of the local day two days before the target date (the last complete day)',
        'cadence': 'Online, on request',
        'horizon': '2 days',
    },
})

## Success Criteria

The output is a temperature, so MAE in degrees is the clearest measure and is less pulled by rare extreme days than RMSE, which is reported as a secondary metric with the bias. MAPE does not suit temperatures near 0. The V1 bar compares with the best simple baseline, which includes persistence from the last complete day; 10% or more leads to promoting the model, less leads to the 'matches persistence' action.

*The offline metric that decides whether V1 is good enough, the bar as a number against a reference, and metrics that must not get worse. `v1_bar` is the bar in words, for people. `v1_criteria` is the same bar as checks a script can apply, each with an `id`, a `kind`, and `on` (cv or test): `beats_model` (`model`, optional `margin`), `pct_over_reference` (`pct`), `threshold` (`metric`, `op`, `value`), `check` (`function` in `src/v1_checks.py`, for project guardrails), or `manual` (`text`, judged by a person). Model training judges the cv criteria and offline evaluation the test criteria.*

In [8]:
# Decision
framing.update({
    'success': {
        'primary_metric': 'MAE',
        'secondary_metrics': ['RMSE', 'mean error (bias)'],
        'v1_bar': 'MAE at least 10% lower than the best simple baseline (persistence, seasonal naive, or moving average) on CV, and still lower than it on the test set',
        'guardrails': [],
        'v1_criteria': [
            {'id': 'beats_best_baseline_cv', 'kind': 'pct_over_reference', 'pct': 10, True: 'cv'},
            {'id': 'beats_best_baseline_test', 'kind': 'pct_over_reference', 'pct': 0, True: 'test'},
        ],
    },
})

## Data Settings and Excluded Columns

No column is recorded after the outcome, and there are no identifiers, personal data, or free text. Redundant columns (grid coordinates, country) are left for feature engineering to keep or drop.

*The main date column, other ID columns, and the columns that must never be features, each with a reason: recorded after the outcome, personal data, an identifier, or timing unknown.*

In [9]:
# Decision
data_settings = {
    'date_col': 'date',
    'id_cols': [],
    'excluded_cols': [],
}

## Checks Against the Data

Each check loads only what it needs and prints counts, never rows. Change `DATA_PATH` if the framing is built on another file, and `TEST_DAYS` to the length of the test period you expect to hold out.

In [10]:
# Decision
DATA_PATH = 'data/raw/open_meteo_daily_2024-01-01_2026-10-01.csv'
TEST_DAYS = 182

In [11]:
df = load_data(DATA_PATH)
target = framing.get("target") or {}
pt = framing.get("prediction_time") or {}
ids = [c for c in framing.get("entity_id_cols") or []]
print(f"{DATA_PATH}: {len(df):,} rows and {df.shape[1]} columns")

data/raw/open_meteo_daily_2024-01-01_2026-10-01.csv: 50,250 rows and 23 columns


### 1. Target

The target column exists, and its class balance (for classification) or spread (for regression).

In [12]:
name = target.get("name")
if target.get("exists_in_data") is False:
    print(f"The target is built, not a column: {target.get('construction') or 'no construction rule yet'}")
elif not name or name not in df.columns:
    print(f"Problem: the target column {name!r} is not in the data.")
elif "classification" in str(framing.get("task_type")):
    shares = df[name].value_counts(normalize=True, dropna=False)
    print(f"{name}: {df[name].nunique()} classes. Shares: {shares.round(4).to_dict()}")
    print(f"Missing targets: {df[name].isna().mean():.2%}")
    if framing.get("task_type") == "binary_classification" and df[name].nunique() != 2:
        print("Problem: binary classification needs exactly two classes.")
else:
    print(df[name].describe().round(3).to_string())

count    50250.000
mean        22.801
std          9.937
min        -25.500
25%         16.900
50%         23.800
75%         30.200
max         47.600


### 2. Unit of Prediction

Rows should be unique on the entity ID columns plus the prediction time column. If they are not, either aggregate them or the unit is finer than the framing says.

In [13]:
key = [c for c in ids + [pt.get("column")] if c]
missing_cols = [c for c in key if c not in df.columns]
if not key:
    print("No entity ID or prediction time column is set yet.")
elif missing_cols:
    print(f"Problem: not in the data: {missing_cols}")
else:
    dup = df.duplicated(subset=key, keep=False).sum()
    print(f"Rows sharing their {' + '.join(key)} with another row: {dup:,} of {len(df):,}")
    for c in ids:
        print(f"{c}: {df[c].nunique():,} values, {len(df) / max(df[c].nunique(), 1):.1f} rows each on average")

Rows sharing their city + date with another row: 0 of 50,250
city: 50 values, 1005.0 rows each on average


### 3. Prediction Time and History

The prediction time column's range, and how much history has complete labels once the outcome window (or the label delay, if longer) is taken off, and the test period is held out.

In [14]:
def to_days(text):
    """Days in a phrase such as '60 days after the scoring date' or '8 weeks'. None when there is no number."""
    m = re.search(r"(\d+(?:\.\d+)?)\s*(day|week|month|year)", str(text or ""), re.I)
    if not m:
        return None
    return float(m.group(1)) * {"day": 1, "week": 7, "month": 30.4, "year": 365}[m.group(2).lower()]


col = pt.get("column")
if not col or col not in df.columns:
    print(f"Problem: the prediction time column {col!r} is not set or not in the data.")
else:
    d = pd.to_datetime(df[col], errors="coerce", format="mixed")
    history = (d.max() - d.min()).days
    window, delay = to_days(target.get("outcome_window")), to_days(target.get("label_delay"))
    # The label is known once the outcome window has passed, or later when there is a label delay. The delay is
    # counted from the prediction, so it includes the window: take the larger of the two.
    labelled = history - max(window or 0, delay or 0)
    print(f"{col}: {d.min():%Y-%m-%d} to {d.max():%Y-%m-%d}, {history:,} days. Unreadable dates: {d.isna().mean():.1%}")
    print(f"Outcome window: {window or 'not set'} days. Label delay: {delay or 'not set'} days.")
    print(f"Labelled history: about {labelled:,.0f} days. After a {TEST_DAYS} day test period: about {labelled - TEST_DAYS:,.0f} days for training.")
    if labelled - TEST_DAYS < 90:
        print("Problem: less than about three months is left for training. Consider a shorter window or test period, or more history.")

date: 2024-01-01 to 2026-10-01, 1,004 days. Unreadable dates: 0.0%
Outcome window: not set days. Label delay: 1.0 days.
Labelled history: about 1,003 days. After a 182 day test period: about 821 days for training.


### 4. Excluded and Risky Columns

Excluded columns should exist. Columns the data dictionary marks as possibly recorded after the outcome, or with unknown timing, should be excluded unless their timing has been confirmed.

In [15]:
excluded = {e["column"] for e in data_settings.get("excluded_cols") or []}
not_found = sorted(excluded - set(df.columns))
print(f"Excluded columns: {len(excluded)}. Not in this file: {not_found or 'none'}")

dict_path = os.path.join(PATHS["docs"], "data_dictionary.yaml")
if os.path.exists(dict_path):
    dictionary = ml_common.load(dict_path)
    main_cols = next((b["columns"] for b in dictionary["sources"].values()
                      if os.path.normpath(b["path"]) == os.path.normpath(DATA_PATH)), [])
    risky = [e["column"] for e in main_cols
             if e["timing"] in ("possibly_after_outcome", "unknown") and e["column"] not in excluded
             and e["column"] != target.get("name")]
    print(f"Kept but possibly recorded after the outcome, or with unknown timing: {risky or 'none'}")
else:
    print("No data dictionary yet. Run 02_data_discovery.ipynb first.")

Excluded columns: 0. Not in this file: none
Kept but possibly recorded after the outcome, or with unknown timing: none


### 5. Metric Against the Class Balance

Accuracy is misleading when one class is rare.

In [16]:
metric = str((framing.get("success") or {}).get("primary_metric") or "")
name = target.get("name")
if name in df.columns and "classification" in str(framing.get("task_type")):
    smallest = df[name].value_counts(normalize=True).min()
    print(f"Smallest class: {smallest:.1%}. Primary metric: {metric or 'not set'}")
    if smallest < 0.05 and "accuracy" in metric.lower():
        print("Problem: the positive class is rare, so prefer PR AUC or recall at k over accuracy.")
else:
    print(f"Primary metric: {metric or 'not set'}")

Primary metric: MAE


## Risks, Assumptions, Open Questions, and Decisions

Risks found when the framing was checked, and how each was resolved. `resolved_questions` closes open questions in the spec that this framing answers, by ID.

In [17]:
# Decision
risks = [
    {
        'problem': 'The first draft used date as the last complete day with a shifted target, which later stages would not guard.',
        'resolution': 'Switched to the forecasting layout (target date rows, horizon 2 days), which the feature and baseline stages handle natively.',
    },
]

assumptions = ['Persistence from the last complete day is the naive forecast with a 2 day lag.']

open_questions = [
    {
        'question': 'How long should the test period be? 90 days covers only one season; a longer period covers more weather but leaves less history for CV.',
        'stage': 'split',
    },
]

decisions = [
    {
        'decision': 'Forecasting layout with target temperature_2m_max and a horizon of 2 days.',
        'reason': 'Requests arrive when the current day is incomplete, so inputs stop at the last complete day.',
    },
    {
        'decision': 'MAE is the primary metric; the V1 bar is 10% under the best simple baseline on CV and still under it on test.',
        'reason': 'The user chose it; it matches the 10% in the answer actions.',
    },
]

resolved_questions = []

## Status

`confirmed` once you agree with the framing and the checks above show no problem you have not accepted. A confirmed framing, followed by `/ml-eda-report`, unlocks modelling.

In [18]:
# Decision
STATUS = 'confirmed'

## Write to the Spec

Writes the `framing` and `data` sections of `ml_spec.yaml`, closes the resolved questions, and records this run.

In [19]:
assert STATUS in ("draft", "confirmed"), "STATUS must be draft or confirmed"
for e in data_settings.get("excluded_cols") or []:
    assert e.get("column") and e.get("reason"), "every excluded column needs a column and a reason"

result = nb_common.write_stage(SPEC_PATH, "framing", {**framing, "risks": risks, "status": STATUS}, "ml-problem-framing",
                               assumptions, open_questions, decisions)
nb_common.write_stage(SPEC_PATH, "data", data_settings, "ml-problem-framing")
not_found = nb_common.resolve_questions(SPEC_PATH, resolved_questions, "ml-problem-framing")
if not_found:
    print("These open question IDs were not found: " + ", ".join(map(str, not_found)))
nb_common.show_result(*result, STATUS)
nb_common.record_run(SPEC_PATH, STAGE, NOTEBOOK, "ml-problem-framing")

Written to ml_spec.yaml with status confirmed.


## Next Step

Run `/ml-eda-report`. It takes the target, date, and ID columns from the spec, analyses the target, checks for leakage, and checks the data against this framing. Check progress with `/ml-status`.